## 1.1 Carga del dataset final y definición de variables

En esta sección cargamos el dataset ya etiquetado (multi‑clase) y definimos la variable objetivo (`y`) y las variables predictoras (`X`). También revisamos la estructura general del conjunto de datos.


In [3]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/ibm_aml_multiclass_clases.csv")

df = pd.read_csv(DATA_PATH)
print("Shape del dataset:", df.shape)
print(df.head())

# Definimos el target multi-clase
TARGET_COL = "target_multi"  

# Seleccionamos un subconjunto razonable de features tabulares para empezar
FEATURE_COLS = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
]

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].astype(int)

print("\nColumnas de features:", FEATURE_COLS)
print("Distribución inicial del target:")
print(y.value_counts().sort_index())


C:\Users\gipas\AppData\Local\Temp\ipykernel_14776\3864789920.py:6: DtypeWarning: Columns (0: pattern_base) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


Shape del dataset: (31898238, 13)
          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:17         20  800104D70       20  800104D70   
1  2022/09/01 00:02       3196  800107150     3196  800107150   
2  2022/09/01 00:17       1208  80010E430     1208  80010E430   
3  2022/09/01 00:03       1208  80010E650       20  80010E6F0   
4  2022/09/01 00:02       1208  80010E650       20  80010EA30   

   Amount Received Receiving Currency  Amount Paid Payment Currency  \
0          6794.63          US Dollar      6794.63        US Dollar   
1          7739.29          US Dollar      7739.29        US Dollar   
2          1880.23          US Dollar      1880.23        US Dollar   
3      73966883.00          US Dollar  73966883.00        US Dollar   
4      45868454.00          US Dollar  45868454.00        US Dollar   

  Payment Format  Is Laundering pattern_base  target_multi  
0   Reinvestment              0          NaN             0  
1   Reinvestment          

## 1.2 Análisis de tipos y datos faltantes

Analizamos los tipos de datos, buscamos valores perdidos y justificamos si es necesario aplicar imputación según las recomendaciones vistas en teoría (Unit 3: tipos de missingness e imputación).


In [4]:
# Tipos de datos
print("Tipos de datos en X:")
print(X.dtypes)

# Análisis de valores perdidos
missing = X.isnull().sum().sort_values(ascending=False)
print("\nNúmero de valores perdidos por columna:")
print(missing)

total_missing = missing.sum()
print(f"\nTotal de valores perdidos en X: {total_missing}")

# Si quieres ver porcentaje
missing_pct = (missing / len(X) * 100).round(6)
print("\nPorcentaje de valores perdidos por columna:")
print(missing_pct[missing_pct > 0])


Tipos de datos en X:
From Bank               int64
Account                   str
To Bank                 int64
Account.1                 str
Amount Received       float64
Receiving Currency        str
Amount Paid           float64
Payment Currency          str
Payment Format            str
dtype: object

Número de valores perdidos por columna:
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
dtype: int64

Total de valores perdidos en X: 0

Porcentaje de valores perdidos por columna:
Series([], dtype: float64)


## 1.3 Selección de variables numéricas y categóricas y estrategia de codificación

- Variables numéricas: identificadores de banco y cuentas, cantidades monetarias.
- Variables categóricas de baja cardinalidad: divisas y `Payment Format`.
- Variables categóricas de alta cardinalidad: identificadores de cuenta (`Account`, `Account.1`).

Para evitar explotar en dimensionalidad, usamos **OrdinalEncoder** en cuentas (muchas categorías)
y **OneHotEncoder** solo en las categóricas de baja cardinalidad.
Las variables numéricas se escalan con `StandardScaler` para estabilizar algunos modelos.


In [5]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Definimos columnas por tipo
num_cols = ["From Bank", "To Bank", "Amount Received", "Amount Paid"]
high_card_cat_cols = ["Account", "Account.1"]  # muchas categorías
low_card_cat_cols = ["Receiving Currency", "Payment Currency", "Payment Format"]

print("Columnas numéricas:", num_cols)
print("Categóricas alta cardinalidad:", high_card_cat_cols)
print("Categóricas baja cardinalidad:", low_card_cat_cols)

# Pipelines parciales
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

high_card_cat_transformer = Pipeline(steps=[
    ("ord_enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

low_card_cat_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ColumnTransformer general de preprocesado
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("high_card_cat", high_card_cat_transformer, high_card_cat_cols),
        ("low_card_cat", low_card_cat_transformer, low_card_cat_cols),
    ]
)

print("\nPreprocesador ColumnTransformer definido.")


Columnas numéricas: ['From Bank', 'To Bank', 'Amount Received', 'Amount Paid']
Categóricas alta cardinalidad: ['Account', 'Account.1']
Categóricas baja cardinalidad: ['Receiving Currency', 'Payment Currency', 'Payment Format']

Preprocesador ColumnTransformer definido.


## 1.4 Aplicación del preprocesado a una muestra

Comprobamos que el `ColumnTransformer` se ajusta correctamente a los datos y observamos
la dimensionalidad del espacio de características transformado.


In [6]:
# Para que sea rápido, usamos una muestra pequeña al probar
X_sample = X.sample(n=50_000, random_state=42)  # ajusta n según tu RAM

X_trans = preprocess.fit_transform(X_sample)
print("Shape original X_sample:", X_sample.shape)
print("Shape transformado:", X_trans.shape)


Shape original X_sample: (50000, 9)
Shape transformado: (50000, 43)


esto se puede borrar -->


In [9]:
for col in ['Account', 'Account.1', 'Receiving Currency', 'Payment Currency', 'Payment Format']:
    print(col, "→", X[col].nunique(), "categorías")


Account → 2013627 categorías
Account.1 → 1689925 categorías
Receiving Currency → 15 categorías
Payment Currency → 15 categorías
Payment Format → 7 categorías


In [10]:
X[num_cols].describe()


,From Bank,To Bank,Amount Received,Amount Paid
count,3.189824e+07,3.189824e+07,3.189824e+07,3.189824e+07
mean,2.944094e+05,4.093198e+05,6.431116e+06,4.417551e+06
std,6.153149e+05,6.547003e+05,2.592744e+09,1.848313e+09
min,0.000000e+00,0.000000e+00,1.000000e-06,1.000000e-06
25%,2.954000e+03,2.749600e+04,2.078700e+02,2.092300e+02
50%,3.902400e+04,1.468530e+05,1.469250e+03,1.471540e+03
75%,2.158840e+05,2.598930e+05,1.183530e+04,1.175781e+04
max,3.225455e+06,3.225455e+06,8.158609e+12,8.158609e+12
